# YAIB Mortality24 baseline validation

This notebook validates the existing 1-hour YAIB mortality24 cohorts for HiRID and MIMIC-IV. It is intentionally read-only with respect to the cohort data: it scans the processed files when executed and writes only derived validation tables and figures.

All analyses use the processed values before YAIB model preprocessing or imputation. Raw measurements, ricu caches, and extraction scripts are not loaded.

In [1]:
from pathlib import Path
import math
import re

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = lambda value: value

DATA_ROOT = Path("/heinz-georgenas/users/mingzhul/Simultaneous-EHR/yaib/data")
RESULTS_DIR = Path("validation_results/mortality24_baseline")
SEED = 42
SAMPLE_STAYS = 100
EXPECTED_STEP_HOURS = 1.0
EXPECTED_WINDOW = (0.0, 24.0)
TIME_TOLERANCE = 1e-9

DATASETS = {
    "HiRID": {"source": "hirid"},
    "MIMIC-IV": {"source": "miiv"},
}
TABLE_FILES = {
    "mortality24": {"dyn": "dyn.parquet", "outc": "outc.parquet", "sta": "sta.parquet", "attrition": "attrition.csv"},
    "base": {"dyn": "dyn.parquet", "sta": "sta.parquet", "attrition": "attrition.csv"},
}

STAY_ID = "stay_id"
TIME = "time"
LABEL = "label"
PATIENT_ID_CANDIDATES = ("patient_id", "subject_id", "patientid", "person_id", "pid")

# Source-defined YAIB variables, declared once here for schema checks.
EXPECTED_DYNAMIC = [
    "alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd",
    "bun", "ca", "cai", "ck", "ckmb", "cl", "crea", "crp", "dbp", "fgn",
    "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact", "lymph", "map",
    "mch", "mchc", "mcv", "methb", "mg", "na", "neut", "o2sat", "pco2",
    "ph", "phos", "plt", "po2", "ptt", "resp", "sbp", "temp", "tnt",
    "urine", "wbc",
]
EXPECTED_STATIC = ["age", "sex", "height", "weight"]
CORE_VARIABLES = ["hr", "sbp", "dbp", "map", "resp", "temp", "o2sat", "glu", "crea", "lact"]
NONNEGATIVE_VARIABLES = {
    "hr", "sbp", "dbp", "map", "resp", "o2sat", "glu", "crea", "lact",
    "alb", "alp", "alt", "ast", "bili", "bun", "ca", "ck", "ckmb", "cl",
    "fio2", "hgb", "k", "lymph", "mch", "mchc", "mcv", "methb", "mg", "na",
    "neut", "phos", "plt", "po2", "ptt", "tnt", "urine", "wbc",
}

plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 11, "axes.labelsize": 9})

## Source contract

The notebook follows the current source definitions:

- YAIB-cohorts writes 'dyn.parquet', 'sta.parquet', 'outc.parquet', and 'attrition.csv'.
- The mortality task uses sequential one-hour windows capped at 24 hours, with task-specific exclusions for death within 30 hours and ICU length of stay below 30 hours.
- YAIB expects 'stay_id', 'time', and 'label' as the group, sequence, and outcome columns.
- The runtime checks below verify the actual processed representation, including whether the window contains timestamps 0 through 24.

In [2]:
def _count(lazy_frame):
    return int(lazy_frame.select(pl.len()).collect().item())

def _unique_count(lazy_frame, column):
    return int(lazy_frame.select(pl.col(column).n_unique()).collect().item())

def _schema(lazy_frame):
    return dict(lazy_frame.collect_schema())

def _is_numeric_dtype(dtype):
    return dtype in {
        pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64, pl.Decimal,
    }

def _base_dtype(dtype):
    return dtype.base_type() if hasattr(dtype, "base_type") else dtype

def _time_hours_expr(column, dtype):
    base = _base_dtype(dtype)
    if base == pl.Datetime:
        return pl.col(column).dt.epoch(time_unit="s").cast(pl.Float64) / 3600.0
    if base == pl.Date:
        return pl.col(column).cast(pl.Int64).cast(pl.Float64) * 24.0
    return pl.col(column).cast(pl.Float64)

def _finite_expr(column):
    return pl.col(column).cast(pl.Float64).is_finite()

def _missing_expr(column):
    return pl.col(column).is_null() | pl.col(column).cast(pl.Float64).is_nan()

def _as_float(value):
    if value is None:
        return float("nan")
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")

def _series_summary(series):
    series = series.drop_nulls()
    if len(series) == 0:
        return {"min": None, "mean": None, "median": None, "p5": None, "p95": None, "max": None}
    return {
        "min": series.min(),
        "mean": series.mean(),
        "median": series.median(),
        "p5": series.quantile(0.05),
        "p95": series.quantile(0.95),
        "max": series.max(),
    }

def _safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_").lower()

def _value(value):
    if value is None:
        return ""
    if isinstance(value, (float, np.floating)) and not np.isfinite(value):
        return ""
    return str(value)

def _write_csv(frame, filename):
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    frame.write_csv(RESULTS_DIR / filename)

In [3]:
def inspect_context():
    context = {}
    missing_files = []

    for dataset, config in DATASETS.items():
        source = config["source"]
        context[dataset] = {"source": source, "paths": {}, "scans": {}, "schemas": {}}
        for cohort, files in TABLE_FILES.items():
            cohort_dir = DATA_ROOT / cohort / source
            context[dataset]["paths"][cohort] = {}
            for table, filename in files.items():
                path = cohort_dir / filename
                context[dataset]["paths"][cohort][table] = path
                if not path.exists():
                    missing_files.append(str(path))
                    continue
                scan = pl.scan_csv(path) if path.suffix == ".csv" else pl.scan_parquet(path)
                context[dataset]["scans"][(cohort, table)] = scan
                context[dataset]["schemas"][(cohort, table)] = _schema(scan)

    if missing_files:
        raise FileNotFoundError(
            "Required processed files were not found. This notebook expects the user to execute it "
            "where the configured YAIB data paths are available:\n" + "\n".join(missing_files)
        )

    required_columns = {
        ("mortality24", "dyn"): [STAY_ID, TIME],
        ("mortality24", "outc"): [STAY_ID, LABEL],
        ("mortality24", "sta"): [STAY_ID],
        ("base", "dyn"): [STAY_ID, TIME],
        ("base", "sta"): [STAY_ID],
    }
    schema_failures = []
    for dataset, info in context.items():
        for (cohort, table), required in required_columns.items():
            available = set(info["schemas"][(cohort, table)])
            absent = sorted(set(required) - available)
            if absent:
                schema_failures.append(f"{dataset} {cohort}/{table}: missing {absent}")
    if schema_failures:
        raise ValueError("Required processed-table columns are missing:\n" + "\n".join(schema_failures))

    return context

# This is the first cell that accesses processed data when the notebook is executed.
ctx = inspect_context()
print("Loaded source metadata for:", ", ".join(ctx))

Loaded source metadata for: HiRID, MIMIC-IV


In [4]:
def _scan(context, dataset, cohort="mortality24", table="dyn"):
    return context[dataset]["scans"][(cohort, table)]

def _anti_count(left, right):
    left_ids = left.select(STAY_ID).unique()
    right_ids = right.select(STAY_ID).unique()
    return _count(left_ids.join(right_ids, on=STAY_ID, how="anti"))

def _label_values(outcome):
    values = outcome.select(LABEL).drop_nulls().unique().collect()[LABEL].to_list()
    return values

def _attrition_rows(context, dataset):
    path = context[dataset]["paths"]["mortality24"]["attrition"]
    frame = pl.scan_csv(path).collect()
    rows = []
    for row in frame.to_dicts():
        for metric, value in row.items():
            rows.append({"dataset": dataset, "metric": metric, "value": _value(value)})
    return rows

def build_overview(context):
    summary_rows = []
    attrition_rows = []
    schema_rows = []

    for dataset in DATASETS:
        info = context[dataset]
        dyn = _scan(context, dataset)
        outc = _scan(context, dataset, table="outc")
        sta = _scan(context, dataset, table="sta")
        base_dyn = _scan(context, dataset, cohort="base")
        base_sta = _scan(context, dataset, cohort="base")

        dynamic_columns = [
            column for column in info["schemas"][("mortality24", "dyn")]
            if column not in {STAY_ID, TIME}
        ]
        static_columns = [
            column for column in info["schemas"][("mortality24", "sta")]
            if column != STAY_ID
        ]
        patient_columns = [
            column for column in PATIENT_ID_CANDIDATES
            if column in info["schemas"][("mortality24", "sta")]
        ]

        label_values = _label_values(outc)
        label_counts = outc.group_by(LABEL).agg(pl.len().alias("count")).collect()
        positives = int(
            label_counts.filter(pl.col(LABEL).cast(pl.Float64) == 1).select("count").sum().item()
            or 0
        )
        negatives = int(
            label_counts.filter(pl.col(LABEL).cast(pl.Float64) == 0).select("count").sum().item()
            or 0
        )
        label_conflicts = _count(
            outc.group_by(STAY_ID).agg(pl.col(LABEL).n_unique().alias("n_labels"))
            .filter(pl.col("n_labels") > 1)
        )
        outc_stays = _unique_count(outc, STAY_ID)
        prevalence = positives / outc_stays if outc_stays else float("nan")

        for table_name, scan in {
            "mortality24_dyn": dyn,
            "mortality24_outc": outc,
            "mortality24_sta": sta,
            "base_dyn": base_dyn,
            "base_sta": base_sta,
        }.items():
            summary_rows.append({
                "dataset": dataset,
                "metric": f"{table_name}_rows",
                "value": _value(_count(scan)),
            })
            summary_rows.append({
                "dataset": dataset,
                "metric": f"{table_name}_unique_stays",
                "value": _value(_unique_count(scan, STAY_ID)),
            })

        for metric, value in {
            "positive_count": positives,
            "negative_count": negatives,
            "mortality_prevalence": prevalence,
            "label_values": ",".join(map(str, sorted(label_values, key=str))),
            "label_conflicting_stays": label_conflicts,
            "dynamic_variable_count": len(dynamic_columns),
            "static_variable_count": len(static_columns),
            "patient_id_columns": ",".join(patient_columns) or "unavailable",
            "patient_count": (
                _unique_count(sta.drop_nulls(patient_columns[0]), patient_columns[0])
                if patient_columns else None
            ),
            "dynamic_columns": ",".join(dynamic_columns),
            "static_columns": ",".join(static_columns),
            "outcome_ids_missing_from_dynamic": _anti_count(outc, dyn),
            "outcome_ids_missing_from_static": _anti_count(outc, sta),
            "dynamic_ids_missing_from_outcome": _anti_count(dyn, outc),
            "static_ids_missing_from_outcome": _anti_count(sta, outc),
        }.items():
            summary_rows.append({"dataset": dataset, "metric": metric, "value": _value(value)})

        expected_dynamic = set(EXPECTED_DYNAMIC)
        observed_dynamic = set(dynamic_columns)
        schema_rows.extend([
            {"dataset": dataset, "schema_group": "dynamic", "status": "missing_expected",
             "variable": variable}
            for variable in sorted(expected_dynamic - observed_dynamic)
        ])
        schema_rows.extend([
            {"dataset": dataset, "schema_group": "dynamic", "status": "unexpected_observed",
             "variable": variable}
            for variable in sorted(observed_dynamic - expected_dynamic)
        ])
        attrition_rows.extend(_attrition_rows(context, dataset))

    overview = pl.DataFrame(summary_rows, schema={
        "dataset": pl.String, "metric": pl.String, "value": pl.String,
    })
    attrition = pl.DataFrame(attrition_rows, schema={
        "dataset": pl.String, "metric": pl.String, "value": pl.String,
    })
    schema = pl.DataFrame(schema_rows, schema={
        "dataset": pl.String, "schema_group": pl.String,
        "status": pl.String, "variable": pl.String,
    }) if schema_rows else pl.DataFrame(
        {"dataset": [], "schema_group": [], "status": [], "variable": []}
    )

    _write_csv(overview, "dataset_summary.csv")
    _write_csv(attrition, "cohort_attrition_summary.csv")
    display(overview)
    display(attrition)
    display(schema)
    return overview, attrition, schema

dataset_summary, attrition_summary, schema_summary = build_overview(ctx)

dataset,metric,value
str,str,str
"""HiRID""","""mortality24_dyn_rows""","""321475"""
"""HiRID""","""mortality24_dyn_unique_stays""","""12859"""
"""HiRID""","""mortality24_outc_rows""","""12859"""
"""HiRID""","""mortality24_outc_unique_stays""","""12859"""
"""HiRID""","""mortality24_sta_rows""","""12859"""
…,…,…
"""MIMIC-IV""","""static_columns""","""age,sex,height,weight"""
"""MIMIC-IV""","""outcome_ids_missing_from_dynam…","""0"""
"""MIMIC-IV""","""outcome_ids_missing_from_stati…","""0"""


dataset,metric,value
str,str,str
"""HiRID""","""incl_n""","""31581"""
"""HiRID""","""excl_n_total""","""932"""
"""HiRID""","""excl_n""","""757"""
"""HiRID""","""incl_n""","""12859"""
"""HiRID""","""excl_n_total""","""20011"""
…,…,…
"""MIMIC-IV""","""excl_n_total""","""1679"""
"""MIMIC-IV""","""excl_n""","""1440"""
"""MIMIC-IV""","""incl_n""","""49523"""


dataset,schema_group,status,variable
null,null,null,null


## Time-grid validation

The source uses a sequential hourly grid. This section measures the actual processed timestamps rather than assuming the grid is correct. Timestamp values are normalized to hours for spacing checks; original timestamp columns remain untouched.

In [5]:
def build_time_artifacts(context):
    artifacts = {}
    summary_rows = []

    for dataset in DATASETS:
        dyn = _scan(context, dataset)
        dyn_schema = context[dataset]["schemas"][("mortality24", "dyn")]
        time_expr = _time_hours_expr(TIME, dyn_schema[TIME]).alias("_time_hours")
        timed = (
            dyn.select([STAY_ID, TIME])
            .with_columns(time_expr)
            .with_columns(
                (pl.col("_time_hours") - pl.col("_time_hours").min().over(STAY_ID)).alias("_offset_hours")
            )
        )
        per_stay = (
            timed.group_by(STAY_ID)
            .agg([
                pl.len().alias("sequence_length"),
                pl.col(TIME).n_unique().alias("unique_timestamps"),
                pl.col("_time_hours").min().alias("min_timestamp_hours"),
                pl.col("_time_hours").max().alias("max_timestamp_hours"),
                pl.col("_offset_hours").min().alias("min_offset_hours"),
                pl.col("_offset_hours").max().alias("max_offset_hours"),
            ])
            .with_columns(
                (pl.col("max_offset_hours") - pl.col("min_offset_hours") + 1.0)
                .alias("expected_bins_from_bounds")
            )
            .with_columns([
                (pl.col("unique_timestamps") == pl.col("sequence_length")).alias("unique_timestamp_ok"),
                (pl.col("unique_timestamps") == pl.col("expected_bins_from_bounds"))
                .alias("complete_span_ok"),
                (
                    (pl.col("min_offset_hours") - EXPECTED_WINDOW[0]).abs() <= TIME_TOLERANCE
                ).alias("starts_at_expected_offset"),
                (
                    (pl.col("max_offset_hours") - EXPECTED_WINDOW[1]).abs() <= TIME_TOLERANCE
                ).alias("ends_at_expected_offset"),
            ])
        )

        with_gaps = (
            timed.sort([STAY_ID, "_offset_hours"])
            .with_columns(pl.col("_offset_hours").diff().over(STAY_ID).alias("_gap_hours"))
        )
        gap_summary = with_gaps.select([
            pl.col("_gap_hours").drop_nulls().len().alias("observed_gaps"),
            (pl.col("_gap_hours").drop_nulls() - EXPECTED_STEP_HOURS).abs()
            .gt(TIME_TOLERANCE).sum().alias("non_hourly_gaps"),
            pl.col("_gap_hours").drop_nulls().min().alias("gap_min_hours"),
            pl.col("_gap_hours").drop_nulls().mean().alias("gap_mean_hours"),
            pl.col("_gap_hours").drop_nulls().median().alias("gap_median_hours"),
            pl.col("_gap_hours").drop_nulls().quantile(0.05).alias("gap_p5_hours"),
            pl.col("_gap_hours").drop_nulls().quantile(0.95).alias("gap_p95_hours"),
            pl.col("_gap_hours").drop_nulls().max().alias("gap_max_hours"),
            (pl.col("_gap_hours").drop_nulls() > EXPECTED_STEP_HOURS + TIME_TOLERANCE)
            .sum().alias("gaps_larger_than_one_hour"),
        ]).collect().to_dicts()[0]

        duplicates = (
            dyn.group_by([STAY_ID, TIME])
            .agg(pl.len().alias("pair_count"))
            .filter(pl.col("pair_count") > 1)
            .collect()
        )
        duplicate_pair_count = len(duplicates)
        duplicate_row_count = int(
            duplicates.select((pl.col("pair_count") - 1).sum()).item() or 0
        )

        per_stay_frame = per_stay.collect()
        artifacts[dataset] = {
            "timed": timed,
            "per_stay": per_stay_frame,
            "with_gaps": with_gaps,
            "duplicate_pair_count": duplicate_pair_count,
            "duplicate_row_count": duplicate_row_count,
            "gap_summary": gap_summary,
        }

        for metric, value in {
            "sequence_length_min": per_stay_frame["sequence_length"].min(),
            "sequence_length_mean": per_stay_frame["sequence_length"].mean(),
            "sequence_length_median": per_stay_frame["sequence_length"].median(),
            "sequence_length_p5": per_stay_frame["sequence_length"].quantile(0.05),
            "sequence_length_p95": per_stay_frame["sequence_length"].quantile(0.95),
            "sequence_length_max": per_stay_frame["sequence_length"].max(),
            "unique_timestamps_min": per_stay_frame["unique_timestamps"].min(),
            "unique_timestamps_max": per_stay_frame["unique_timestamps"].max(),
            "minimum_timestamp_hours": per_stay_frame["min_timestamp_hours"].min(),
            "maximum_timestamp_hours": per_stay_frame["max_timestamp_hours"].max(),
            "minimum_offset_hours": per_stay_frame["min_offset_hours"].min(),
            "maximum_offset_hours": per_stay_frame["max_offset_hours"].max(),
            "duplicate_timestamp_pairs": duplicate_pair_count,
            "duplicate_rows_beyond_first": duplicate_row_count,
            "observed_gaps": gap_summary["observed_gaps"],
            "non_hourly_gaps": gap_summary["non_hourly_gaps"],
            "gaps_larger_than_one_hour": gap_summary["gaps_larger_than_one_hour"],
            "gap_min_hours": gap_summary["gap_min_hours"],
            "gap_mean_hours": gap_summary["gap_mean_hours"],
            "gap_median_hours": gap_summary["gap_median_hours"],
            "gap_p5_hours": gap_summary["gap_p5_hours"],
            "gap_p95_hours": gap_summary["gap_p95_hours"],
            "gap_max_hours": gap_summary["gap_max_hours"],
            "stays_with_25_timestamps": per_stay_frame.filter(
                pl.col("unique_timestamps") == 25
            ).height,
            "stays_with_expected_window": per_stay_frame.filter(
                pl.col("starts_at_expected_offset") & pl.col("ends_at_expected_offset")
            ).height,
            "stays_with_incomplete_span": per_stay_frame.filter(
                ~pl.col("complete_span_ok")
            ).height,
        }.items():
            summary_rows.append({
                "dataset": dataset,
                "metric": metric,
                "value": _value(value),
            })

    result = pl.DataFrame(summary_rows, schema={
        "dataset": pl.String, "metric": pl.String, "value": pl.String,
    })
    _write_csv(result, "time_grid_summary.csv")
    display(result)
    return result, artifacts

time_grid_summary, time_artifacts = build_time_artifacts(ctx)

dataset,metric,value
str,str,str
"""HiRID""","""sequence_length_min""","""25"""
"""HiRID""","""sequence_length_mean""","""25.0"""
"""HiRID""","""sequence_length_median""","""25.0"""
"""HiRID""","""sequence_length_p5""","""25.0"""
"""HiRID""","""sequence_length_p95""","""25.0"""
…,…,…
"""MIMIC-IV""","""gap_p95_hours""","""3600000.0"""
"""MIMIC-IV""","""gap_max_hours""","""3600000.0"""
"""MIMIC-IV""","""stays_with_25_timestamps""","""49523"""


## Missingness and measurement density

The processed dynamic table contains one row per generated hourly bin, not raw measurement events. Therefore, an observed bin means that the processed variable is nonmissing in that bin; raw measurement counts are not inferred.

In [6]:
def _numeric_dynamic_columns(context, dataset):
    schema = context[dataset]["schemas"][("mortality24", "dyn")]
    return [
        column for column, dtype in schema.items()
        if column not in {STAY_ID, TIME} and _is_numeric_dtype(dtype)
    ]

def build_missingness_artifacts(context, time_artifacts):
    missing_rows = []
    observation_rows = []
    availability_rows = []
    density_rows = []
    artifacts = {}

    for dataset in DATASETS:
        variables = _numeric_dynamic_columns(context, dataset)
        dyn = _scan(context, dataset)
        time_schema = context[dataset]["schemas"][("mortality24", "dyn")][TIME]

        obs = (
            dyn.with_columns(_time_hours_expr(TIME, time_schema).alias("_time_hours"))
            .with_columns(
                (pl.col("_time_hours") - pl.col("_time_hours").min().over(STAY_ID))
                .alias("_offset_hours")
            )
            .select([STAY_ID, "_offset_hours"] + [
                (~_missing_expr(variable)).alias(variable) for variable in variables
            ])
            .with_columns(
                pl.sum_horizontal(variables).alias("_observed_variables")
            )
            .cache()
        )

        row_counts = obs.select([
            pl.len().alias("_rows"),
            (pl.col("_observed_variables") == 0).sum().alias("_empty_rows"),
        ] + [
            pl.col(variable).sum().alias(variable) for variable in variables
        ]).collect().to_dicts()[0]

        per_stay = obs.group_by(STAY_ID).agg([
            pl.col(variable).sum().alias(variable) for variable in variables
        ]).collect()

        for variable in variables:
            observed_bins = int(row_counts[variable] or 0)
            total_rows = int(row_counts["_rows"] or 0)
            missing_bins = total_rows - observed_bins
            values = per_stay[variable]
            summary = _series_summary(values)
            observed_gaps = (
                obs.filter(pl.col(variable))
                .select([STAY_ID, "_offset_hours"])
                .sort([STAY_ID, "_offset_hours"])
                .with_columns(
                    pl.col("_offset_hours").diff().over(STAY_ID).alias("_observed_gap")
                )
                .select("_observed_gap")
                .drop_nulls()
                .collect()["_observed_gap"]
            )
            gap_stats = _series_summary(observed_gaps)
            stays_with_observation = int((values > 0).sum())
            total_stays = per_stay.height
            missing_rows.append({
                "dataset": dataset,
                "variable": variable,
                "total_processed_bins": total_rows,
                "missing_bin_count": missing_bins,
                "nonmissing_bin_count": observed_bins,
                "missing_percentage": (100.0 * missing_bins / total_rows) if total_rows else None,
                "nonmissing_percentage": (100.0 * observed_bins / total_rows) if total_rows else None,
                "stays_with_observation": stays_with_observation,
                "stays_entirely_missing": total_stays - stays_with_observation,
                "stays_with_observation_percentage": (
                    100.0 * stays_with_observation / total_stays if total_stays else None
                ),
            })
            observation_rows.append({
                "dataset": dataset,
                "variable": variable,
                "observed_bins_min": summary["min"],
                "observed_bins_median": summary["median"],
                "observed_bins_p5": summary["p5"],
                "observed_bins_p95": summary["p95"],
                "observed_bins_max": summary["max"],
                "observed_gap_min_hours": gap_stats["min"],
                "observed_gap_median_hours": gap_stats["median"],
                "observed_gap_p5_hours": gap_stats["p5"],
                "observed_gap_p95_hours": gap_stats["p95"],
                "observed_gap_max_hours": gap_stats["max"],
            })

        availability = (
            obs.group_by("_offset_hours")
            .agg([
                pl.len().alias("total_bins"),
                (pl.col("_observed_variables") > 0).sum().alias("bins_with_any_observation"),
                pl.col("_observed_variables").mean().alias("mean_observed_variables"),
            ])
            .sort("_offset_hours")
            .collect()
        )
        for row in availability.to_dicts():
            total = row["total_bins"]
            availability_rows.append({
                "dataset": dataset,
                "time_offset_hours": row["_offset_hours"],
                "total_bins": total,
                "bins_with_any_observation": row["bins_with_any_observation"],
                "availability_percentage": (
                    100.0 * row["bins_with_any_observation"] / total if total else None
                ),
                "mean_observed_variables": row["mean_observed_variables"],
            })

        density_rows.append({
            "dataset": dataset,
            "total_processed_bins": row_counts["_rows"],
            "entirely_empty_bins": row_counts["_empty_rows"],
            "entirely_empty_bin_percentage": (
                100.0 * row_counts["_empty_rows"] / row_counts["_rows"]
                if row_counts["_rows"] else None
            ),
            "mean_observed_variables_per_bin": (
                obs.select(pl.col("_observed_variables").mean()).collect().item()
            ),
        })
        artifacts[dataset] = {
            "variables": variables,
            "obs": obs,
            "per_stay_observation_counts": per_stay,
            "availability": availability,
        }

    missingness = pl.DataFrame(missing_rows)
    observation_counts = pl.DataFrame(observation_rows)
    availability_df = pl.DataFrame(availability_rows)
    density = pl.DataFrame(density_rows)

    _write_csv(missingness, "variable_missingness.csv")
    _write_csv(observation_counts, "variable_observation_counts.csv")
    display(missingness)
    display(observation_counts)
    display(density)
    return missingness, observation_counts, availability_df, density, artifacts

missingness, observation_counts, availability, density, missingness_artifacts = (
    build_missingness_artifacts(ctx, time_artifacts)
)

# Add density metrics to the overview output without recomputing dynamic scans.
density_overview_rows = []
for row in density.to_dicts():
    for metric in [
        "total_processed_bins", "entirely_empty_bins",
        "entirely_empty_bin_percentage", "mean_observed_variables_per_bin",
    ]:
        density_overview_rows.append({
            "dataset": row["dataset"],
            "metric": metric,
            "value": _value(row[metric]),
        })
density_overview = pl.DataFrame(density_overview_rows)
dataset_summary = pl.concat([dataset_summary, density_overview], how="vertical_relaxed")
_write_csv(dataset_summary, "dataset_summary.csv")
display(density_overview)
display(dataset_summary)

dataset,variable,total_processed_bins,missing_bin_count,nonmissing_bin_count,missing_percentage,nonmissing_percentage,stays_with_observation,stays_entirely_missing,stays_with_observation_percentage
str,str,i64,i64,i64,f64,f64,i64,i64,f64
"""HiRID""","""alb""",321475,320286,1189,99.630142,0.369858,1064,11795,8.27436
"""HiRID""","""alp""",321475,317223,4252,98.677347,1.322653,3449,9410,26.821681
"""HiRID""","""alt""",321475,311463,10012,96.885605,3.114395,7770,5089,60.424605
"""HiRID""","""ast""",321475,312252,9223,97.131037,2.868963,7109,5750,55.284237
"""HiRID""","""be""",321475,261647,59828,81.389533,18.610467,11674,1185,90.784664
…,…,…,…,…,…,…,…,…,…
"""MIMIC-IV""","""sbp""",1238075,121089,1116986,9.780425,90.219575,49339,184,99.628455
"""MIMIC-IV""","""temp""",1238075,863383,374692,69.735921,30.264079,47911,1612,96.744947
"""MIMIC-IV""","""tnt""",1238075,1216638,21437,98.268522,1.731478,10658,38865,21.521313


dataset,variable,observed_bins_min,observed_bins_median,observed_bins_p5,observed_bins_p95,observed_bins_max,observed_gap_min_hours,observed_gap_median_hours,observed_gap_p5_hours,observed_gap_p95_hours,observed_gap_max_hours
str,str,i64,f64,f64,f64,i64,f64,f64,f64,f64,f64
"""HiRID""","""alb""",0,0.0,0.0,1.0,3,3.6e6,4.32e7,1.8e7,7.56e7,8.64e7
"""HiRID""","""alp""",0,0.0,0.0,2.0,5,3.6e6,3.96e7,1.44e7,7.92e7,8.64e7
"""HiRID""","""alt""",0,1.0,0.0,2.0,6,3.6e6,3.6e7,1.44e7,7.92e7,8.64e7
"""HiRID""","""ast""",0,1.0,0.0,2.0,6,3.6e6,3.6e7,1.44e7,7.92e7,8.64e7
"""HiRID""","""be""",0,5.0,0.0,10.0,21,3.6e6,1.08e7,3.6e6,3.24e7,8.64e7
…,…,…,…,…,…,…,…,…,…,…,…
"""MIMIC-IV""","""sbp""",0,24.0,16.0,25.0,25,3.6e6,3.6e6,3.6e6,3.6e6,8.64e7
"""MIMIC-IV""","""temp""",0,7.0,3.0,17.0,25,3.6e6,1.44e7,3.6e6,1.8e7,8.64e7
"""MIMIC-IV""","""tnt""",0,0.0,0.0,3.0,7,3.6e6,2.52e7,1.08e7,5.04e7,8.64e7


dataset,total_processed_bins,entirely_empty_bins,entirely_empty_bin_percentage,mean_observed_variables_per_bin
str,i64,i64,f64,f64
"""HiRID""",321475,7963,2.47702,10.173031
"""MIMIC-IV""",1238075,32426,2.619066,8.868945


dataset,metric,value
str,str,str
"""HiRID""","""total_processed_bins""","""321475"""
"""HiRID""","""entirely_empty_bins""","""7963"""
"""HiRID""","""entirely_empty_bin_percentage""","""2.477019986002022"""
"""HiRID""","""mean_observed_variables_per_bi…","""10.17303056225212"""
"""MIMIC-IV""","""total_processed_bins""","""1238075"""
"""MIMIC-IV""","""entirely_empty_bins""","""32426"""
"""MIMIC-IV""","""entirely_empty_bin_percentage""","""2.619065888577025"""
"""MIMIC-IV""","""mean_observed_variables_per_bi…","""8.868944934676817"""


dataset,metric,value
str,str,str
"""HiRID""","""mortality24_dyn_rows""","""321475"""
"""HiRID""","""mortality24_dyn_unique_stays""","""12859"""
"""HiRID""","""mortality24_outc_rows""","""12859"""
"""HiRID""","""mortality24_outc_unique_stays""","""12859"""
"""HiRID""","""mortality24_sta_rows""","""12859"""
…,…,…
"""HiRID""","""mean_observed_variables_per_bi…","""10.17303056225212"""
"""MIMIC-IV""","""total_processed_bins""","""1238075"""
"""MIMIC-IV""","""entirely_empty_bins""","""32426"""


## Missingness figures

In [7]:
def plot_missingness_figures(missingness, observation_counts, availability, artifacts):
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    variables = sorted(missingness["variable"].unique().to_list())
    datasets = list(DATASETS)
    matrix = np.array([
        [
            float(missingness.filter(
                (pl.col("dataset") == dataset) & (pl.col("variable") == variable)
            )["missing_percentage"][0] or np.nan)
            if missingness.filter(
                (pl.col("dataset") == dataset) & (pl.col("variable") == variable)
            ).height else np.nan
            for variable in variables
        ]
        for dataset in datasets
    ])

    fig, ax = plt.subplots(figsize=(max(10, len(variables) * 0.28), 2.8))
    image = ax.imshow(matrix, aspect="auto", cmap="magma", vmin=0, vmax=100)
    ax.set_yticks(range(len(datasets)), datasets)
    ax.set_xticks(range(len(variables)), variables, rotation=90, fontsize=7)
    ax.set_title("Dynamic-variable missingness by hospital")
    fig.colorbar(image, ax=ax, label="Missing processed bins (%)")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "missingness_heatmap.png", bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(max(10, len(variables) * 0.28), 4.0))
    for dataset in datasets:
        frame = missingness.filter(pl.col("dataset") == dataset).sort("variable")
        ax.plot(frame["variable"].to_list(), frame["missing_percentage"].to_list(), marker=".", label=dataset)
    ax.set_ylabel("Missing processed bins (%)")
    ax.set_title("Missingness percentage by variable")
    ax.tick_params(axis="x", rotation=90)
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "missingness_percentage.png", bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(max(10, len(variables) * 0.28), 4.0))
    for dataset in datasets:
        frame = observation_counts.filter(pl.col("dataset") == dataset).sort("variable")
        ax.plot(frame["variable"].to_list(), frame["observed_bins_median"].to_list(), marker=".", label=dataset)
    ax.set_ylabel("Median observed hourly bins per stay")
    ax.set_title("Observed-bin counts by variable")
    ax.tick_params(axis="x", rotation=90)
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "observed_bin_counts.png", bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4))
    for dataset in datasets:
        frame = availability.filter(pl.col("dataset") == dataset).sort("time_offset_hours")
        ax.plot(frame["time_offset_hours"], frame["availability_percentage"], marker=".", label=dataset)
    ax.set_xlabel("Hours from first processed timestamp")
    ax.set_ylabel("Bins with any observation (%)")
    ax.set_title("Observation availability across the processed window")
    ax.set_xlim(*EXPECTED_WINDOW)
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "observation_availability.png", bbox_inches="tight")
    plt.close(fig)

    for dataset in datasets:
        dyn = _scan(ctx, dataset)
        variables = missingness_artifacts[dataset]["variables"]
        selected = (
            dyn.select(STAY_ID).unique().sort(STAY_ID)
            .collect()
            .sample(n=min(SAMPLE_STAYS, _unique_count(dyn, STAY_ID)), seed=SEED)
        )
        selected_ids = selected[STAY_ID].to_list()
        schema = ctx[dataset]["schemas"][("mortality24", "dyn")]
        time_expr = _time_hours_expr(TIME, schema[TIME]).alias("_time_hours")
        sample = (
            dyn.filter(pl.col(STAY_ID).is_in(selected_ids))
            .with_columns(time_expr)
            .with_columns(
                (pl.col("_time_hours") - pl.col("_time_hours").min().over(STAY_ID))
                .alias("_offset_hours")
            )
            .select([STAY_ID, "_offset_hours"] + variables)
            .collect()
        )
        offsets = list(range(int(EXPECTED_WINDOW[0]), int(EXPECTED_WINDOW[1]) + 1))
        matrix = np.zeros((len(selected_ids), len(offsets)), dtype=float)
        for i, stay_id in enumerate(selected_ids):
            rows = sample.filter(pl.col(STAY_ID) == stay_id)
            for row in rows.iter_rows(named=True):
                offset = row["_offset_hours"]
                if offset is None:
                    continue
                offset_int = int(round(offset))
                if offset_int in offsets:
                    observed = any(
                        row[variable] is not None and np.isfinite(float(row[variable]))
                        for variable in variables
                    )
                    matrix[i, offsets.index(offset_int)] = float(observed)

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap="gray_r", vmin=0, vmax=1)
        ax.set_xlabel("Hour offset")
        ax.set_ylabel("Fixed-seed sampled stays")
        ax.set_title(f"{dataset}: observation mask for {len(selected_ids)} stays")
        ax.set_xticks(range(len(offsets)), offsets)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / f"{_safe_name(dataset)}_observation_masks.png", bbox_inches="tight")
        plt.close(fig)

plot_missingness_figures(missingness, observation_counts, availability, missingness_artifacts)

## Feature-value distributions

Statistics use finite, nonmissing values directly from processed dynamic tables, before YAIB scaling, forward fill, zero fill, or model-based imputation. Histograms use a deterministic sample when a variable has more than 100,000 values; plotted axes use the pooled 1st–99th percentile range for readability while the CSV statistics retain the full finite range.

In [8]:
def build_value_statistics(context):
    rows = []
    stats_by_dataset = {}

    for dataset in DATASETS:
        variables = _numeric_dynamic_columns(context, dataset)
        dyn = _scan(context, dataset)
        expressions = []
        for variable in variables:
            finite = _finite_expr(variable) & ~pl.col(variable).is_null()
            values = pl.col(variable).cast(pl.Float64).filter(finite)
            all_values = pl.col(variable).cast(pl.Float64)
            expressions.extend([
                values.count().alias(f"{variable}__count"),
                values.mean().alias(f"{variable}__mean"),
                values.std().alias(f"{variable}__std"),
                values.min().alias(f"{variable}__min"),
                values.quantile(0.01).alias(f"{variable}__p1"),
                values.quantile(0.05).alias(f"{variable}__p5"),
                values.quantile(0.25).alias(f"{variable}__p25"),
                values.median().alias(f"{variable}__median"),
                values.quantile(0.75).alias(f"{variable}__p75"),
                values.quantile(0.95).alias(f"{variable}__p95"),
                values.quantile(0.99).alias(f"{variable}__p99"),
                values.max().alias(f"{variable}__max"),
                (~_missing_expr(variable)).sum().alias(f"{variable}__nonmissing"),
                (~_finite_expr(variable) & ~pl.col(variable).is_null()).sum()
                .alias(f"{variable}__nonfinite"),
                (all_values < 0).sum().alias(f"{variable}__negative"),
            ])
        wide = dyn.select(expressions).collect().to_dicts()[0]
        stats_by_dataset[dataset] = wide
        for variable in variables:
            rows.append({
                "dataset": dataset,
                "variable": variable,
                **{
                    metric: wide.get(f"{variable}__{metric}")
                    for metric in [
                        "count", "mean", "std", "min", "p1", "p5", "p25",
                        "median", "p75", "p95", "p99", "max", "nonmissing",
                        "nonfinite", "negative",
                    ]
                },
            })

    result = pl.DataFrame(rows)
    _write_csv(result, "variable_value_statistics.csv")
    display(result)
    return result, stats_by_dataset

value_statistics, value_stats_by_dataset = build_value_statistics(ctx)

def plot_value_distributions(context):
    all_variables = sorted(set().union(*[
        set(_numeric_dynamic_columns(context, dataset)) for dataset in DATASETS
    ]))
    for variable in all_variables:
        samples = {}
        combined = []
        for dataset in DATASETS:
            if variable not in _numeric_dynamic_columns(context, dataset):
                continue
            values = (
                _scan(context, dataset)
                .select(pl.col(variable).cast(pl.Float64))
                .drop_nulls()
                .collect()[variable]
                .to_numpy()
            )
            values = values[np.isfinite(values)]
            if len(values) > 100_000:
                values = np.random.default_rng(SEED).choice(values, 100_000, replace=False)
            samples[dataset] = values
            combined.append(values)
        if not combined:
            continue
        if not combined or not any(len(values) for values in combined):
            continue
        pooled = np.concatenate(combined)
        if len(pooled) == 0:
            continue
        lo, hi = np.quantile(pooled, [0.01, 0.99])
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            lo, hi = float(np.min(pooled)), float(np.max(pooled))
        if lo == hi:
            lo, hi = lo - 0.5, hi + 0.5
        bins = np.linspace(lo, hi, 41)

        fig, ax = plt.subplots(figsize=(6.5, 4))
        for dataset, values in samples.items():
            shown = values[(values >= lo) & (values <= hi)]
            ax.hist(shown, bins=bins, density=True, alpha=0.45, label=dataset)
        ax.set_title(f"{variable}: processed value distribution")
        ax.set_xlabel(variable)
        ax.set_ylabel("Density")
        ax.legend()
        ax.text(
            0.99, 0.01, "Displayed range: pooled p1–p99",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=7,
        )
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / f"value_distribution_{_safe_name(variable)}.png", bbox_inches="tight")
        plt.close(fig)

plot_value_distributions(ctx)

dataset,variable,count,mean,std,min,p1,p5,p25,median,p75,p95,p99,max,nonmissing,nonfinite,negative
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64
"""HiRID""","""alb""",1189,2.203028,0.636812,0.7,0.9,1.2,1.8,2.2,2.7,3.2,3.7,4.1,1189,0,0
"""HiRID""","""alp""",4252,107.487065,147.436471,9.0,22.0,30.0,46.0,68.0,113.0,299.0,676.0,2408.0,4252,0,0
"""HiRID""","""alt""",10012,239.169347,762.291992,3.0,6.0,10.0,19.0,36.0,102.0,1197.0,3937.0,11595.0,10012,0,0
"""HiRID""","""ast""",9223,473.379486,1687.426354,4.0,12.0,16.0,28.0,59.0,189.0,2099.0,8600.0,34760.0,9223,0,0
"""HiRID""","""be""",59828,-1.735349,4.863138,-25.0,-16.3,-10.3,-4.2,-1.3,1.1,5.4,10.1,24.9,59828,0,38079
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""MIMIC-IV""","""sbp""",1116986,117.715534,21.086796,0.0,79.0,88.0,102.0,115.0,131.0,156.0,176.0,282.0,1116986,0,0
"""MIMIC-IV""","""temp""",374692,36.90407,0.715267,32.0,35.0,35.833333,36.555556,36.888889,37.222222,38.111111,39.0,42.0,374692,0,0
"""MIMIC-IV""","""tnt""",21437,0.933572,2.337492,0.0,0.01,0.02,0.05,0.17,0.66,4.62,12.66,51.84,21437,0,0


## Validation summary

In [9]:
def add_check(rows, dataset, check, status, observed, expected, detail=""):
    rows.append({
        "dataset": dataset,
        "check": check,
        "status": status,
        "observed": _value(observed),
        "expected": _value(expected),
        "detail": detail,
    })

def build_validation_checks(context, time_artifacts, missingness, value_statistics, schema_summary):
    rows = []
    expected_dynamic = set(EXPECTED_DYNAMIC)

    for dataset in DATASETS:
        info = context[dataset]
        dyn_schema = info["schemas"][("mortality24", "dyn")]
        outc = _scan(context, dataset, table="outc")
        dyn = _scan(context, dataset)
        sta = _scan(context, dataset, table="sta")
        observed_dynamic = set(
            column for column in dyn_schema if column not in {STAY_ID, TIME}
        )
        time_info = time_artifacts[dataset]
        per_stay = time_info["per_stay"]
        time_summary = time_info["gap_summary"]

        add_check(rows, dataset, "required_table_files", "PASS", "present", "present")
        add_check(
            rows, dataset, "required_columns", "PASS",
            "stay_id/time/label present", "stay_id/time/label present",
        )

        labels = _label_values(outc)
        binary = all(_as_float(label) in {0.0, 1.0} for label in labels)
        add_check(
            rows, dataset, "binary_labels", "PASS" if binary else "FAIL",
            labels, "{0, 1}",
            "Mortality labels must be encoded as 0/1.",
        )

        conflicting = _count(
            outc.group_by(STAY_ID).agg(pl.col(LABEL).n_unique().alias("n_labels"))
            .filter(pl.col("n_labels") > 1)
        )
        add_check(
            rows, dataset, "one_label_per_stay", "PASS" if conflicting == 0 else "FAIL",
            conflicting, 0,
        )

        duplicates = time_info["duplicate_pair_count"]
        add_check(
            rows, dataset, "duplicate_stay_time_pairs",
            "PASS" if duplicates == 0 else "FAIL",
            duplicates, 0,
        )

        irregular = int(time_summary["non_hourly_gaps"] or 0)
        add_check(
            rows, dataset, "hourly_spacing",
            "PASS" if irregular == 0 else "FAIL",
            irregular, 0,
            "Checks consecutive processed timestamps after normalization to hours.",
        )

        incomplete = per_stay.filter(~pl.col("complete_span_ok")).height
        add_check(
            rows, dataset, "complete_timestamp_spans",
            "PASS" if incomplete == 0 else "FAIL",
            incomplete, 0,
        )

        out_of_window = per_stay.filter(
            ~pl.col("starts_at_expected_offset") | ~pl.col("ends_at_expected_offset")
        ).height
        add_check(
            rows, dataset, "observation_window_0_to_24",
            "PASS" if out_of_window == 0 else "FAIL",
            out_of_window, 0,
            "The source task caps the sequential mortality window at 24 hours.",
        )

        non25 = per_stay.filter(pl.col("unique_timestamps") != 25).height
        add_check(
            rows, dataset, "inclusive_25_timestamp_window",
            "PASS" if non25 == 0 else "WARN",
            non25, 0,
            "WARN means runtime data should be reviewed for endpoint semantics.",
        )

        missing_expected = sorted(expected_dynamic - observed_dynamic)
        schema_status = "PASS" if not missing_expected else "WARN"
        add_check(
            rows, dataset, "expected_dynamic_schema",
            schema_status, ",".join(missing_expected) or "none", "none",
            "Schema differences can reflect hospital-specific source availability.",
        )

        missing_ids = sum([
            _anti_count(outc, dyn), _anti_count(outc, sta),
            _anti_count(dyn, outc), _anti_count(sta, outc),
        ])
        add_check(
            rows, dataset, "stay_id_consistency", "PASS" if missing_ids == 0 else "FAIL",
            missing_ids, 0,
        )

        nonfinite = int(
            value_statistics.filter(pl.col("dataset") == dataset)["nonfinite"]
            .fill_null(0).sum()
        )
        add_check(
            rows, dataset, "nonfinite_dynamic_values",
            "PASS" if nonfinite == 0 else "FAIL",
            nonfinite, 0,
        )

        suspicious = int(
            value_statistics.filter(
                (pl.col("dataset") == dataset) &
                pl.col("variable").is_in(list(NONNEGATIVE_VARIABLES))
            )["negative"].fill_null(0).sum()
        )
        add_check(
            rows, dataset, "negative_values_in_nonnegative_variables",
            "PASS" if suspicious == 0 else "WARN",
            suspicious, 0,
            "This is a broad diagnostic warning, not a clinical plausibility threshold.",
        )

        for variable in sorted(expected_dynamic & observed_dynamic):
            other_dataset = next(
                name for name in DATASETS if name != dataset
            )
            if variable in _numeric_dynamic_columns(context, other_dataset):
                own = missingness.filter(
                    (pl.col("dataset") == dataset) & (pl.col("variable") == variable)
                )["missing_percentage"][0]
                other = missingness.filter(
                    (pl.col("dataset") == other_dataset) & (pl.col("variable") == variable)
                )["missing_percentage"][0]
                if own is not None and other is not None and abs(float(own) - float(other)) > 50:
                    add_check(
                        rows, dataset, f"missingness_difference_{variable}", "WARN",
                        own, other,
                        f"Difference versus {other_dataset}; hospital differences are expected.",
                    )

    result = pl.DataFrame(rows)
    _write_csv(result, "validation_checks.csv")
    display(result)

    status_counts = result.group_by("status").agg(pl.len().alias("count")).sort("status")
    display(status_counts)
    return result

validation_checks = build_validation_checks(
    ctx, time_artifacts, missingness, value_statistics, schema_summary
)

dataset,check,status,observed,expected,detail
str,str,str,str,str,str
"""HiRID""","""required_table_files""","""PASS""","""present""","""present""",""""""
"""HiRID""","""required_columns""","""PASS""","""stay_id/time/label present""","""stay_id/time/label present""",""""""
"""HiRID""","""binary_labels""","""PASS""","""[1, 0]""","""{0, 1}""","""Mortality labels must be encod…"
"""HiRID""","""one_label_per_stay""","""PASS""","""0""","""0""",""""""
"""HiRID""","""duplicate_stay_time_pairs""","""PASS""","""0""","""0""",""""""
…,…,…,…,…,…
"""MIMIC-IV""","""inclusive_25_timestamp_window""","""PASS""","""0""","""0""","""WARN means runtime data should…"
"""MIMIC-IV""","""expected_dynamic_schema""","""PASS""","""none""","""none""","""Schema differences can reflect…"
"""MIMIC-IV""","""stay_id_consistency""","""PASS""","""0""","""0""",""""""


status,count
str,u32
"""FAIL""",6
"""PASS""",20


## Processing notes and future frequency validation

The cohort source constructs sequential patient windows and maps processed concepts onto the generated grid. The current mortality task uses a one-hour frequency and caps observation at 24 hours. ricu performs concept-level transformations and interval aggregation; the extension dictionaries provide explicit callbacks and selected aggregation declarations for concepts such as urine output, temperature conversion, outcomes, and medications. The notebook does not infer raw measurement-event counts from the processed tables.

YAIB preprocessing is downstream of these files. Its classification preprocessor can scale features, add missingness indicators, forward-fill dynamic variables, and then zero-fill remaining gaps. Optional model-based imputation is configured separately. This notebook therefore reports the pre-imputation processed values and missingness, before those training-time operations.

When HiRID is regenerated at 5-minute resolution with fixed cohort selection, the following should remain invariant: stay IDs, mortality labels, positive counts, prevalence, physical observation-window duration, and feature definitions. Sequence lengths, time-bin counts, missingness percentages, observed-bin counts, and aggregated value distributions are expected to change. Cross-frequency comparisons are intentionally not implemented here.